# Maya (Maya-Q1): healthcare operations LLM — train + eval

Non-clinical hospital operations only. No diagnosis, treatment, prescription or triage.

Model: `empero-ai/Qwen3.8-4B-Distill` (Qwen3.5 architecture, Apache-2.0).

Training path: **plain Transformers QLoRA (4-bit)**. Unsloth is NOT used for training on a T4
(it forces float32 for Qwen3.5 because the T4 lacks bf16, which OOMs). 4-bit bitsandbytes fits.

On a bf16-capable GPU (A100/L4/RTX 30xx+), the notebook auto-detects bf16 and uses bf16 LoRA.

## 1. Clone the repository

In [ ]:
import os, sys
REPO = os.environ.get("MAYA_REPO", "https://github.com/Cyrax321/maya-healthcare-llm.git")
!rm -rf maya-healthcare-llm
!git clone -b main {REPO}
%cd maya-healthcare-llm
sys.path.insert(0, os.path.abspath('src'))  # make the maya package importable right away

## 2. Install dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q "transformers==5.15.0" accelerate peft trl bitsandbytes datasets
!pip install -q -e . --no-deps
import sys, os
sys.path.insert(0, os.path.abspath('src'))
import maya
print('maya importable:', maya.__version__)

## 3. GPU / software diagnostics

In [ ]:
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('transformers', transformers.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print('device:', torch.cuda.get_device_name(0), '| capability:', cap)
    print('bf16 supported:', cap[0] >= 8)

## 4. Prepare the dataset (deterministic synthetic + safety cases)

In [ ]:
from maya import data
manifest = data.prepare_dataset(seed=20241015, examples_per_category=520, out_dir="data/processed")
print('splits:', manifest['splits'])
print('categories:', manifest['audit']['category_counts'])

## 5. Train (smoke 5 steps -> full 300 steps)

Auto-detects bf16. On a T4 -> 4-bit QLoRA. On Ampere+ -> bf16 LoRA.

In [ ]:
import gc, torch
from maya import config, data, train
from maya.model import DEFAULT_REPO

cfgs = config.load_all_configs()
HAS_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
LOAD_4BIT = not HAS_BF16
print(f'load_in_4bit={LOAD_4BIT} (bf16={HAS_BF16})')

tr = data.load_processed_split("data/processed", "train")
va = data.load_processed_split("data/processed", "validation")

# smoke: 5 steps, proves it fits before the full run
smoke_cfg = config.load_all_configs()
smoke_cfg["training_t4"]["training"]["max_steps"] = 5
res = train.run_training_transformers(DEFAULT_REPO, tr[:64], va[:16], "outputs/smoke", smoke_cfg, load_in_4bit=LOAD_4BIT, resume=False)
print('SMOKE OK:', res.final_checkpoint)
del res; gc.collect(); torch.cuda.empty_cache()

# full train (300 steps from configs/training_t4.yaml)
res = train.run_training_transformers(DEFAULT_REPO, tr, va, "outputs/maya-q1", cfgs, load_in_4bit=LOAD_4BIT, resume=True)
print('FULL TRAIN DONE:', res.final_checkpoint)

## 6. Push adapter to HF + save to Drive (BEFORE eval)

Eval is the slow part; protect the trained adapter first. Set the `HF_TOKEN`
Colab Secret and fix `HUB_USER` to your Hugging Face username.

In [ ]:
import os, shutil
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except ImportError:
    pass  # local machine: export HF_TOKEN in the shell instead
HUB_USER = os.environ.get('HUB_USER', 'Cyrax321')

from huggingface_hub import HfApi
api = HfApi()
api.create_repo(f'{HUB_USER}/Maya-Q1', repo_type='model', private=True, exist_ok=True)
api.upload_folder(folder_path='outputs/maya-q1', repo_id=f'{HUB_USER}/Maya-Q1', repo_type='model')
print('adapter pushed to HF:', f'{HUB_USER}/Maya-Q1')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copytree('outputs', '/content/drive/MyDrive/maya/outputs', dirs_exist_ok=True)
    print('adapter saved to Drive')
except ImportError:
    print('not Colab; skipping Drive')

## 7. Evaluate base (E0) vs adapter (E1) on the locked test set

In [ ]:
import torch
flag = '--load-in-4bit' if not (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else ''
!PYTHONPATH=src python -m maya.cli evaluate --experiment-name E0 --experiment-id E0 {flag}
!PYTHONPATH=src python -m maya.cli evaluate --experiment-name E1 --experiment-id E1 --adapter outputs/maya-q1 {flag}
!PYTHONPATH=src python -m maya.cli compare --baseline artifacts/E0 --adapter artifacts/E1

## 8. Push results to HF + save artifacts to Drive

In [ ]:
api.create_repo(f'{HUB_USER}/Maya-Q1-results', repo_type='dataset', private=True, exist_ok=True)
for exp in ['E0', 'E1']:
    api.upload_folder(folder_path=f'artifacts/{exp}', repo_id=f'{HUB_USER}/Maya-Q1-results', repo_type='dataset', path_in_repo=exp)
print('results pushed to HF:', f'{HUB_USER}/Maya-Q1-results')

try:
    shutil.copytree('artifacts', '/content/drive/MyDrive/maya/artifacts', dirs_exist_ok=True)
    print('artifacts saved to Drive')
except Exception:
    pass